In [1]:
import copy
import math
from contextlib import contextmanager
from operator import attrgetter

import glog
import torch
from torch import multiprocessing as mp
from torch import nn
from transformers import AutoModelForCausalLM
import argparse
import numpy as np
import sys
import os
import time ## ryu
import gc

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', 'qtip'))

if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"Added to path: {project_root}")

try:
    from lib import codebook, utils
    from lib.linear.quantized_linear import QuantizedLinear
    from lib.codebook import bitshift
    from lib.algo import ldlq   
    print("Imports successful!")
except ImportError as e:
    print(f"Import failed: {e}")
    print("Check if the path exists:", os.path.exists(project_root))


def qtip_quantize(quip_params, W, HR, device):    
    rcp = 'col'
    orig_dtype = torch.float32
    dtype_ = torch.float32
    split_for_tp = False
    tp_rank = 8
    scale_override = -1
    
    td_x = quip_params['td_x']
    td_y = quip_params['td_y']
    L = quip_params['L']
    K = quip_params['K']
    V = quip_params['V']
    tlut_bits = quip_params['tlut_bits']
    decode_mode = quip_params['decode_mode']
    
    cb = bitshift.bitshift_codebook(L=L,
                                K=K,
                                V=V,
                                tlut_bits=tlut_bits,
                                decode_mode=decode_mode)
    
    
    has_kernel = utils.has_kernel(decode_mode, L, K, V,
                tlut_bits, td_x, td_y)
    
    cb = cb.to(device).to(orig_dtype)    
    # use_bias = (orig_linear.bias != None) ## ryu
    # bias = orig_linear.bias ## ryu
    # del orig_linear
    (m, n) = W.shape
    SU = (torch.randn(n, device=device).sign() + 1e-5).sign().to(dtype_)
    SV = (torch.randn(m, device=device).sign() + 1e-5).sign().to(dtype_)

    # in_hess_path = f'{in_hess_path}/{idx}_{in_hess_name}.pt'
    # # in_hess_path = f'{in_hess_path}/lang_{idx}_{in_hess_name}.pt'
    # H_data = torch.load(in_hess_path, map_location=torch.device('cpu'))
    # HR = utils.flat_to_sym(H_data['flatH'], H_data['n'])
    # if 'mu' in H_data:
    #     mu = H_data['mu']
    #     HR += mu[None, :] * mu[:, None]
    #     del mu
    # del H_data

    # HR = utils.regularize_H(HR, sigma_reg)

    W = W.to(device)
    HR = HR.to(device)
    torch.cuda.synchronize()
    gc.collect()
    gc.disable()  # GC 비활성화

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    start_event.record()

    if split_for_tp:
        pass
        # if rcp == 'col':
        #     # split along output dimension
        #     Wr = utils.matmul_hadUt(
        #         utils.matmul_hadUt((W.T.to(device) * SV).reshape(
        #             n * tp_rank, m // tp_rank)).reshape(
        #                 W.T.shape).T * SU)
        #     HRr = utils.matmul_hadUt(
        #         utils.matmul_hadUt(HR.to(device) * SU).T * SU)

        #     Wscale = Wr.reshape(
        #         tp_rank, m * n // tp_rank).square().mean(
        #             dim=-1).sqrt() / (cb.lut.to(
        #                 torch.float64).square().mean().sqrt().float() *
        #                                 scale_override)
        #     Wr = Wr.reshape(tp_rank,
        #                     m * n // tp_rank) / Wscale.unsqueeze(-1)
        #     Wr = Wr.reshape(m, n)

        # elif rcp == 'row':
        #     # split along input dimension
        #     Wr = utils.matmul_hadUt(
        #         (utils.matmul_hadUt(W.T.to(device) * SV).T * SU).reshape(
        #             m * tp_rank, n // tp_rank)).reshape(W.shape)
        #     HRr = utils.matmul_hadUt(
        #         (utils.matmul_hadUt((HR.to(device) * SU).reshape(
        #             n * tp_rank, n // tp_rank)).reshape(n, n).T *
        #             SU).reshape(n * tp_rank,
        #                         n // tp_rank)).reshape(n, n)
        #     Wscale = Wr.reshape(
        #         m, tp_rank,
        #         n // tp_rank).transpose(0, 1).reshape(
        #             tp_rank, m * n // tp_rank).square().mean(
        #                 dim=-1).sqrt() / (cb.lut.to(
        #                     torch.float64).square().mean().sqrt().float() *
        #                                     scale_override)
        #     Wr = Wr.reshape(m, tp_rank, n // tp_rank).transpose(
        #         0, 1).reshape(tp_rank,
        #                         m * n // tp_rank) / Wscale.unsqueeze(-1)
        #     Wr = Wr.reshape(tp_rank, m,
        #                     n // tp_rank).transpose(0,
        #                                                     1).reshape(m, n)
    else:
        Wr = utils.matmul_hadUt(
            utils.matmul_hadUt(W.T.to(device) * SV).T * SU)
        HRr = utils.matmul_hadUt(
            utils.matmul_hadUt(HR.to(device) * SU).T * SU)
        
        Wscale = Wr.square().mean().sqrt() / (
            cb.lut.to(torch.float64).square().mean().sqrt().float() *
            scale_override)
        Wr /= Wscale

    LRr, _ = utils.block_LDL(HRr, td_y)
    diag = torch.arange(n, device=LRr.device)
    LRr[diag, diag] = 0

    args = argparse.Namespace(**quip_params)
    hatWr, Qidxs = ldlq.LDLQ(Wr, LRr, cb, args, for_kernel=has_kernel)

    Qidxs = Qidxs.cpu()
    packed = cb.pack_trellis(
        Qidxs.reshape(m // td_x, td_x, n // td_y,
                        td_y // V).transpose(1, 2).reshape(
                            -1, td_x * td_y // V))

    if has_kernel:
        packed = packed.view(torch.uint8).view(-1, 2).flip(
            (-1, )).reshape(m // 16 // 2, 2, n // 16 // 2, 2, 16 * 16 // 8,
                            K).permute(0, 2, 4, 3, 1, 5).flip(
                                (-1, )).contiguous().flatten().view(
                                    torch.int16).reshape(packed.shape)
    else:
        packed = packed.view(torch.int16)

    end_event.record()
    torch.cuda.synchronize()
    gc.enable() # GC 다시 활성화
    elapsed_time_ms = start_event.elapsed_time(end_event)
    elapsed_time = elapsed_time_ms / 1000.0
   
    glog.info(f"Total encoding Time: {elapsed_time_ms*1000:.4f} ms")

    if rcp == 'col':
        Wr = (Wr.reshape(tp_rank, m * n // tp_rank) *
                Wscale.unsqueeze(-1)).reshape(m, n)
        hatWr = (hatWr.reshape(tp_rank, m * n // tp_rank) *
                    Wscale.unsqueeze(-1)).reshape(m, n)
    elif rcp == 'row':
        Wr = Wr.reshape(m, tp_rank, n // tp_rank).transpose(
            0, 1).reshape(tp_rank, -1) * Wscale.unsqueeze(-1)
        Wr = Wr.reshape(tp_rank, m,
                        n // tp_rank).transpose(0, 1).reshape(m, n)
        hatWr = hatWr.reshape(m, tp_rank,
                                n // tp_rank).transpose(0, 1).reshape(
                                    tp_rank, -1) * Wscale.unsqueeze(-1)
        hatWr = hatWr.reshape(tp_rank, m,
                                n // tp_rank).transpose(0, 1).reshape(
                                    m, n)
    else:
        Wr *= Wscale
        hatWr *= Wscale

    err = torch.trace(
        (Wr - hatWr) @ HRr @ (Wr - hatWr).T) / torch.trace(Wr @ HRr @ Wr.T)
    print(
        f'proxy err {err.item()} tr(WHW.T) {torch.trace(Wr @ HRr @ Wr.T)}'
    )

    save_path = f'./tmp_qtip_ckpt.pt'

    # 0 = no tensor parallelism, 1 = row parallel, 2 = column parallel
    rcp_int = 0
    if split_for_tp:
        rcp_int = 1 if rcp == 'row' else 2

    torch.save(
        {
            'trellis':
            packed.cpu(),
            'SU':
            SU.to(orig_dtype).cpu(),
            'SV':
            SV.to(orig_dtype).cpu(),
            'Wscale':
            Wscale,
            'proxy_err':
            err.item(),
            'tr(WHW.T)': torch.trace(Wr @ HRr @ Wr.T).item(),
            'mse': torch.mean((Wr - hatWr) ** 2).item(),
            'tlut':
            cb.tlut.data.to(orig_dtype).cpu()
            if hasattr(cb, 'tlut') else None,
            'rcp':
            rcp_int,
            'tp_rank':
            tp_rank,
            # 'bias':bias, ## ryu
            'time': elapsed_time_ms
        }, save_path)

    del HRr, Wr, hatWr, LRr, Qidxs
    utils.clean()
    return elapsed_time_ms

def initialize_codebook(quant_layer):
    assert not hasattr(quant_layer, 'built_codebook_class') or not quant_layer.built_codebook_class
    quant_layer.codebook_class = bitshift.BitshiftLinear(
        quant_layer.td_x, quant_layer.td_y, quant_layer.L,
        quant_layer.K, quant_layer.V, quant_layer.tlut_bits,
        quant_layer.decode_mode, dtype=quant_layer.dtype,
        tlut=quant_layer.tlut, has_kernel=quant_layer.has_kernel
    )
    rcp = quant_layer.rcp.item()
    del quant_layer.rcp
    quant_layer.rcp = rcp
    quant_layer.built_codebook_class = True

def get_What(quip_params, orig_layer_weight, saved_layer_data, layer_name, device):
    """
    양자화된 데이터를 기반으로 복원된 가중치(W_hat)를 계산하여 반환합니다.
    LLaMA 수정: MoE Router용 예외 처리(td_x // 2)를 제거했습니다.
    """
    td_x = quip_params['td_x']
    td_y = quip_params['td_y']
    L = quip_params['L']
    K = quip_params['K']
    V = quip_params['V']
    tlut_bits = quip_params['tlut_bits']
    decode_mode = quip_params['decode_mode']
    
    # 임시 QuantizedLinear 생성
    quant_layer = QuantizedLinear(orig_layer_weight.shape[1],
                    orig_layer_weight.shape[0],
                    td_x, td_y, L, K, V, tlut_bits, decode_mode,
                    dtype=orig_layer_weight.dtype,
                    bias=True)
    
    quant_layer.mode = 'train-fixW'
    quant_layer.to(device) # 계산은 GPU에서 수행
    utils.unpack_quip(quant_layer, saved_layer_data)
    
    quant_layer.has_kernel = utils.has_kernel(decode_mode, L, K, V, tlut_bits, td_x, td_y)
    initialize_codebook(quant_layer)
    
    torch.cuda.synchronize()
    gc.collect()
    gc.disable()

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    start_event.record()
    quant_layer.codebook_class.cache_hatW(quant_layer.trellis, quant_layer.had_left,
                                       quant_layer.had_right, quant_layer.K_left,
                                       quant_layer.K_right, len(quant_layer.SV),
                                       len(quant_layer.SU), quant_layer.rcp,
                                       quant_layer.tp_rank)

    end_event.record()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    gc.enable()
    elapsed_time_ms = start_event.elapsed_time(end_event)
    
    
    glog.info(f"{layer_name} Total decoding time: {elapsed_time_ms} ms")
    
    hatW = quant_layer.codebook_class.hatW
    SU = quant_layer.SU
    SV = quant_layer.SV
    scale = quant_layer.codebook_class.scale

    target_dtype = hatW.dtype     
    SV = SV.to(target_dtype)
    SU = SU.to(target_dtype)

    W_reconstructed = torch.diag(SV * scale) @ hatW @ torch.diag(SU)

    return elapsed_time_ms

W0104 12:45:59.264022 1236109 warnings.py:109] /opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Added to path: /home/jgryu/workspace/weight_compression/qtip


I0104 12:46:00.352975 1236109 utils.py:148] Note: detected 128 virtual cores but NumExpr set to maximum of 64, check "NUMEXPR_MAX_THREADS" environment variable.
I0104 12:46:00.354203 1236109 utils.py:151] Note: NumExpr detected 128 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
I0104 12:46:00.354764 1236109 utils.py:164] NumExpr defaulting to 16 threads.


Imports successful!


In [2]:
print(f"{'Mode':<15} | {'K':<3} | {'Enc Mean(ms)':<12} | {'Enc Std':<10} | {'Dec Mean(ms)':<12} | {'Dec Std':<10}")
print("-" * 80)

device = 'cuda:0'
num_iter = 10        # 반복 횟수 (통계용)
warmup_iter = 1      # 워밍업 횟수 (캐싱 등 오버헤드 제거)
std_val = 0.012528750114142895
N = 4096             # Matrix Size

# modes = [('1mad', 0), ('3inst', 0), ('quantlut_sym', 9), ('lut', 16)]
modes = [('quantlut_sym', 9)]
K_values = [2, 3, 4]

H = torch.eye(N, device=device) 

for decode_mode, tlut_bits in modes:
    for K in K_values:
        quip_params = {
            'td_x' : 16,
            'td_y' : 16,
            'L' : 16,
            'K' : K,
            'V' : 2,
            'tlut_bits' : tlut_bits,
            'decode_mode' : decode_mode
        }
        
        enc_times = []
        dec_times = []
        
        for i in range(num_iter + warmup_iter):
            W = torch.normal(mean=0.0, std=std_val, size=(N, N), device=device)
            
            encoding_time = qtip_quantize(quip_params, W, H, device)
            saved = torch.load(f'./tmp_qtip_ckpt.pt', map_location='cpu', weights_only=False)
            decoding_time = get_What(quip_params, W, saved, layer_name=" ", device = device)

            if i >= warmup_iter:
                enc_times.append(encoding_time)
                dec_times.append(decoding_time)

        # 통계 계산
        enc_mean = np.mean(enc_times)
        enc_std = np.std(enc_times)
        dec_mean = np.mean(dec_times)
        dec_std = np.std(dec_times)
        
        print(f"{decode_mode:<15} | {K:<3} | {enc_mean:<12.4f} | {enc_std:<10.4f} | {dec_mean:<12.4f} | {dec_std:<10.4f}")

Mode            | K   | Enc Mean(ms) | Enc Std    | Dec Mean(ms) | Dec Std   
--------------------------------------------------------------------------------


W0104 12:46:03.436025 1236109 warnings.py:109] /home/jgryu/workspace/weight_compression/qtip/lib/codebook/bitshift.py:161: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tlut

proxy err 0.07052982598543167 tr(WHW.T) 2634.283935546875


W0104 12:46:28.373000 140277886628480 torch/fx/experimental/symbolic_shapes.py:4449] [1/0] q5 is not in var_ranges, defaulting to unknown range.
W0104 12:46:28.373000 140277886628480 torch/fx/experimental/symbolic_shapes.py:4449] [1/0] q3 is not in var_ranges, defaulting to unknown range.
W0104 12:46:28.374000 140277886628480 torch/fx/experimental/symbolic_shapes.py:4449] [1/0] q6 is not in var_ranges, defaulting to unknown range.
W0104 12:46:28.374000 140277886628480 torch/fx/experimental/symbolic_shapes.py:4449] [1/0] q4 is not in var_ranges, defaulting to unknown range.
W0104 12:46:28.375000 140277886628480 torch/fx/experimental/symbolic_shapes.py:4449] [1/0] q1 is not in var_ranges, defaulting to unknown range.
W0104 12:46:28.375000 140277886628480 torch/fx/experimental/symbolic_shapes.py:4449] [1/0] q0 is not in var_ranges, defaulting to unknown range.
W0104 12:46:28.376000 140277886628480 torch/fx/experimental/symbolic_shapes.py:4449] [1/0] q2 is not in var_ranges, defaulting to 

proxy err 0.07053771615028381 tr(WHW.T) 2632.873046875


...

proxy err 0.07053180038928986 tr(WHW.T) 2633.19921875


I0104 12:50:11.573614 1236109 3670712213.py:299]   Total decoding time: 15.609312057495117 ms
W0104 12:50:11.575583 1236109 warnings.py:109] /home/jgryu/workspace/weight_compression/qtip/lib/codebook/bitshift.py:161: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded fi

quantlut_sym    | 2   | 21070.0539   | 315.6632   | 7.4985       | 5.5055    


  0%|          | 0/32 [00:00<?, ?it/s]


BackendCompilerFailed: backend='inductor' raised:
AssertionError: Failed to find static RBLOCK for PowByNatural(2, 2*s0)

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


You can suppress this exception and fall back to eager by setting:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True


In [ ]:
project_root

'/home/jgryu/workspace/weight_compression/qtip'